# AgentDebug — Google Colab Setup

Runs the AgentDebug pipeline using **Qwen2.5-Coder-7B-Instruct** (4-bit quantized) on Colab's T4 GPU.

**No API key. No cost. Model loads directly on GPU.**

### Instructions
1. Run **Step 1** (install deps) — then **restart runtime** when prompted
2. After restart, run **Step 2 onwards** (skip Step 1 on re-run)

## Step 1: Install Dependencies (run once, then restart runtime)

In [ ]:
# Fix numpy binary incompatibility with Colab's pre-installed packages
!pip install -q numpy==1.26.4

# Core ML dependencies (bitsandbytes needed for 4-bit quantization)
!pip install -q --upgrade transformers accelerate bitsandbytes

# Pipeline dependencies
!pip install -q langchain==0.1.0 langchain-community==0.0.20
!pip install -q python-dotenv pyyaml tqdm seaborn

print("\n" + "="*60)
print("Dependencies installed!")
print("NOW: Go to Runtime > Restart runtime")
print("THEN: Skip this cell and run Step 2 onwards")
print("="*60)

## Step 2: Clone Repo & Verify Data

In [ ]:
import os

# Clone repo (skips if already cloned)
if not os.path.exists('/content/major_project'):
    !git clone https://github.com/Amitanand0123/major_project.git
    print("Repo cloned.")
else:
    print("Repo already exists, pulling latest...")
    !cd /content/major_project && git pull

%cd /content/major_project

# Verify trajectory data came with the repo
traj_dir = 'data/swebench/final_trajectories'
traj_files = [f for f in os.listdir(traj_dir) if f.endswith('.json')]
print(f"Trajectory files: {len(traj_files)}")
print("Ready for Step 3.")

## Step 3: Load Model & Test

Downloads **Qwen2.5-Coder-7B-Instruct** (~4.5 GB) on first run.
Uses **4-bit quantization** — fits easily on T4 with fast inference. Cached after first download.

In [ ]:
import sys, torch
sys.path.append('/content/major_project')

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")

# Load model (4-bit quantized to fit T4)
from run_complete_pipeline import setup_llm
llm = setup_llm(provider='huggingface', model_name='Qwen/Qwen2.5-Coder-7B-Instruct')

# Quick sanity test
test_response = llm.invoke("Say 'AgentDebug ready' if you can read this.")
print(f"\nLLM test: {test_response[:100]}")
print("\nAll checks passed!")

## Step 4: Run a Single Batch (50 trajectories)

In [ ]:
from run_daily_batch import DailyBatchRunner

runner = DailyBatchRunner(base_dir='results_1000_study')
runner.print_overall_progress()

batch_info = await runner.run_daily_batch(
    trajectory_dir='data/swebench/final_trajectories',
    llm=llm
)

if batch_info:
    print(f"\nBatch {batch_info['batch_number']} complete!")
else:
    print("\nBatch failed. Check errors above.")

## Step 5: Run Multiple Batches

Each batch = 50 trajectories. Adjust `NUM_BATCHES` as needed.

In [ ]:
NUM_BATCHES = 4  # 4 batches x 50 = 200 trajectories

runner = DailyBatchRunner(base_dir='results_1000_study')

for i in range(NUM_BATCHES):
    print(f"\n{'='*60}")
    print(f"BATCH {i+1} of {NUM_BATCHES}")
    print(f"{'='*60}")
    
    batch_info = await runner.run_daily_batch(
        trajectory_dir='data/swebench/final_trajectories',
        llm=llm
    )
    
    if batch_info:
        print(f"Batch {batch_info['batch_number']} done!")
    else:
        print("Batch failed, stopping.")
        break

runner.print_overall_progress()

## Step 6: Download Results

Zips and downloads results to your local machine. **Do this before the runtime disconnects!**

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/results_1000_study', 'zip', '.', 'results_1000_study')
print("Results zipped!")

files.download('/content/results_1000_study.zip')
print("Download started!")

## Step 7: Quick Stats

In [ ]:
import json
from pathlib import Path

results_dir = Path('results_1000_study')
progress_file = results_dir / 'progress.json'

if progress_file.exists():
    with open(progress_file) as f:
        progress = json.load(f)
    
    print(f"Total trajectories completed: {progress['total_trajectories_completed']}")
    print(f"Batches completed: {len(progress['completed_batches'])}")
    
    total_results = 0
    for batch_dir in sorted(results_dir.glob('batch_*')):
        for run_dir in batch_dir.glob('run_*'):
            individual_dir = run_dir / 'experiments' / 'individual'
            if individual_dir.exists():
                count = len(list(individual_dir.glob('*_analysis.json')))
                total_results += count
                print(f"  {batch_dir.name}: {count} results")
    
    print(f"\nTotal analysis files: {total_results}")
else:
    print("No results yet.")